# KLTN — V1.3: Regularize + Augment trên 56 đặc trưng hình học

**Mục tiêu:** Cải thiện V1.2 (F1=0.7182) lên ≥ 0.78 (đạt ngưỡng tối thiểu đề cương) bằng:

1. **BiLSTM** thay LSTM thuần — bắt được pattern cả 2 chiều thời gian
2. **Dropout** 0.3/0.5 — chống overfit (V1.2 train_acc=0.90 nhưng val_acc=0.72 → overfit rõ ràng)
3. **Augmentation:** Gaussian noise σ=0.02 + Time crop ngẫu nhiên — tăng đa dạng dữ liệu
4. **Focal Loss** α=0.25, γ=2.0 — giảm dominant của mẫu dễ, ép mô hình học mẫu khó
5. **AdamW** + weight decay — chính quy hoá thêm

**Dự kiến:** F1 0.75-0.82 (vượt ngưỡng 0.78 là thành công).

## Trước khi chạy
- Folder `Human-Reco/processed_data1/` (56 feat) phải có sẵn trên Drive
- Notebook V1.2 đã chạy xong (để so sánh)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT_DIR = "/content/drive/MyDrive/KLTN/Human-Reco"   # ★ Sửa nếu khác
assert os.path.exists(f"{ROOT_DIR}/processed_data1"), "Thiếu processed_data1/"
OUT_DIR = f"{ROOT_DIR}/training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"ROOT = {ROOT_DIR}")

In [ ]:
!pip install -q tensorflow scikit-learn pandas seaborn matplotlib numpy --upgrade

In [ ]:
import os, json, time, glob, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Bidirectional, Conv2D, MaxPooling2D,
                                     GlobalAveragePooling2D, BatchNormalization,
                                     Dense, Reshape, Input, Dropout, Lambda)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
print("TF:", tf.__version__, "GPU:", tf.config.list_physical_devices('GPU'))
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

## 2. Regenerate X 56-feat + clip_ids (như V1.2)

In [ ]:
def regenerate_56feat(processed_root, seq_length=90, step=30):
    X, y, clip_ids = [], [], []
    for label_name in ['normal', 'shoplifting']:
        label_val = 0 if label_name == 'normal' else 1
        folder = os.path.join(processed_root, label_name)
        for fp in sorted(glob.glob(os.path.join(folder, "*.csv"))):
            clip_id = f"{label_name}/{os.path.basename(fp)}"
            data = pd.read_csv(fp).values
            if data.shape[1] != 56: continue
            for i in range(0, len(data) - seq_length + 1, step):
                X.append(data[i: i + seq_length])
                y.append(label_val)
                clip_ids.append(clip_id)
    return (np.array(X, dtype='float32'),
            np.array(y, dtype='int'),
            np.array(clip_ids))

X, y, clip_ids = regenerate_56feat(f"{ROOT_DIR}/processed_data1")
print(f"X.shape = {X.shape},  y.shape = {y.shape},  clips = {len(np.unique(clip_ids))}")

## 3. Split-by-clip (giống V1.1/V1.2)

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
idx_tr, idx_rem = next(gss1.split(X, y, groups=clip_ids))
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED+1)
idx_v_loc, idx_t_loc = next(gss2.split(X[idx_rem], y[idx_rem], groups=clip_ids[idx_rem]))
idx_v = idx_rem[idx_v_loc]; idx_t = idx_rem[idx_t_loc]
X_tr, y_tr = X[idx_tr], y[idx_tr]
X_v, y_v   = X[idx_v], y[idx_v]
X_t, y_t   = X[idx_t], y[idx_t]
assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_v]))
assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_t]))
assert not (set(clip_ids[idx_v])  & set(clip_ids[idx_t]))
print(f"Train: {len(idx_tr)} | Val: {len(idx_v)} | Test: {len(idx_t)}")

## 4. Augmentation pipeline

Áp dụng **chỉ trên tập train**. Hai phép tăng cường:

- **Gaussian noise** σ=0.02 — mô phỏng nhiễu YOLO-Pose
- **Time crop** random — cắt 60-90 frame liên tiếp rồi interpolate về 90, mô phỏng tốc độ hành động khác nhau

Không dùng rotation/flip vì 56 đặc trưng hình học đã được tính trên cấu trúc cố định, biến đổi geometric phức tạp.

In [ ]:
def augment_sample(sample, noise_std=0.02, crop_prob=0.3, training=True):
    """sample shape (90, 56). Áp dụng noise + time crop."""
    if not training:
        return sample
    s = sample.copy()
    # Gaussian noise
    if np.random.rand() < 0.8:
        s = s + np.random.normal(0, noise_std, s.shape).astype(np.float32)
    # Time crop
    if np.random.rand() < crop_prob:
        T = s.shape[0]
        crop_T = np.random.randint(60, T + 1)
        if crop_T < T:
            start = np.random.randint(0, T - crop_T + 1)
            cropped = s[start: start + crop_T]
            # Linear interpolate back to T
            idx_src = np.linspace(0, crop_T - 1, num=T)
            i_floor = np.floor(idx_src).astype(int)
            i_ceil = np.minimum(i_floor + 1, crop_T - 1)
            w = (idx_src - i_floor).astype(np.float32)[:, None]
            s = (1 - w) * cropped[i_floor] + w * cropped[i_ceil]
    return s.astype(np.float32)


class AugDataGen(Sequence):
    def __init__(self, X, y, batch_size=64, augment=True, shuffle=True):
        self.X = X; self.y = y
        self.bs = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        if shuffle:
            np.random.shuffle(self.indices)
    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, idx):
        sl = self.indices[idx*self.bs : (idx+1)*self.bs]
        Xb = np.stack([augment_sample(self.X[i], training=self.augment) for i in sl])
        yb = self.y[sl]
        return Xb, yb
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


train_gen = AugDataGen(X_tr, y_tr, batch_size=64, augment=True,  shuffle=True)
val_gen   = AugDataGen(X_v,  y_v,  batch_size=64, augment=False, shuffle=False)
print(f"Train batches: {len(train_gen)}, Val batches: {len(val_gen)}")
# Test sample
xb, yb = train_gen[0]
print(f"Sample batch X shape: {xb.shape}, y shape: {yb.shape}, dtype: {xb.dtype}")

## 5. Focal Loss

In [ ]:
def focal_loss(alpha=0.25, gamma=2.0):
    """Focal loss cho sparse categorical (y_true là int 0/1)."""
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_oh = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        y_true_oh = tf.cast(y_true_oh, y_pred.dtype)
        y_pred_clamped = tf.clip_by_value(y_pred, 1e-8, 1.0 - 1e-8)
        ce = -y_true_oh * tf.math.log(y_pred_clamped)
        weight = alpha * tf.math.pow(1 - y_pred_clamped, gamma)
        focal = weight * ce
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))
    return loss_fn

## 6. Mô hình V1.3: BiLSTM + Dropout + Conv2D

Thay đổi so V1.2:
- LSTM → BiLSTM(32) (output 64 chiều thay vì 32)
- Thêm Dropout(0.3) sau mỗi BiLSTM
- Thêm Dropout(0.3) sau Conv2D
- Thêm Dropout(0.5) trước Dense cuối
- Reshape thành (90, 64, 1) vì BiLSTM ra 64 chiều

In [ ]:
def build_v13(input_shape=(90, 56), num_classes=2):
    model = Sequential([
        Input(shape=input_shape),
        Bidirectional(LSTM(32, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32, return_sequences=True)),
        Dropout(0.3),
        Reshape((90, 64, 1)),  # BiLSTM x2 trả về 64 features
        Conv2D(64, (5, 5), strides=(2, 2), padding='same', activation='relu'),
        MaxPooling2D((2, 2), strides=(2, 2)),
        Dropout(0.3),
        Conv2D(128, (3, 3), strides=(1, 1), padding='same', activation='relu'),
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=AdamW(learning_rate=1e-3, weight_decay=5e-4),
        loss=focal_loss(alpha=0.25, gamma=2.0),
        metrics=['accuracy'],
    )
    return model

model = build_v13()
model.summary()
print(f"\nTotal params: {model.count_params():,}")

## 7. Train V1.3

In [ ]:
ckpt_path = f"{OUT_DIR}/best_v1.3_56feat_regularized.keras"
cbs = [
    ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, verbose=0),
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1),
]
t0 = time.time()
hist = model.fit(train_gen, validation_data=val_gen,
                 epochs=120, callbacks=cbs, verbose=2)
train_time = time.time() - t0
print(f"\nTrained {len(hist.history['loss'])} epochs in {train_time:.1f}s")

## 8. Đánh giá trên test set

In [ ]:
# Không augment khi predict
y_pred = model.predict(X_t, verbose=0).argmax(1)
f1m = f1_score(y_t, y_pred, average='macro')
report = classification_report(y_t, y_pred, target_names=['Normal','Shoplifting'], digits=4)
cm = confusion_matrix(y_t, y_pred)
print(report)
print(f"Confusion matrix:\n{cm}")
print(f"F1-macro: {f1m:.4f}")

log = {
    "config": "v1.3_56feat_regularized",
    "input_shape": [90, 56],
    "improvements": [
        "BiLSTM thay LSTM",
        "Dropout 0.3/0.3/0.3/0.5",
        "Gaussian noise σ=0.02 + Time crop augmentation",
        "Focal Loss α=0.25 γ=2.0",
        "AdamW + weight_decay 5e-4",
    ],
    "train_size": int(X_tr.shape[0]),
    "val_size": int(X_v.shape[0]),
    "test_size": int(X_t.shape[0]),
    "epochs_trained": len(hist.history['loss']),
    "best_val_loss": float(min(hist.history['val_loss'])),
    "test_f1_macro": float(f1m),
    "test_confusion_matrix": cm.tolist(),
    "test_classification_report": report,
    "train_time_sec": train_time,
    "total_params": int(model.count_params()),
    "history": {k: [float(v) for v in vs] for k, vs in hist.history.items()},
}
with open(f"{OUT_DIR}/log_v1.3_56feat_regularized.json", "w") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)
print(f"\n✓ Saved {OUT_DIR}/log_v1.3_56feat_regularized.json")

## 9. So sánh V1.1 → V1.2 → V1.3

In [ ]:
results = {"V1.3 — Regularized": f1m}
for name, fn in [("V1.1 — 34 feat", "log_v1.1_split_by_clip.json"),
                 ("V1.2 — 56 feat", "log_v1.2_56feat_by_clip.json")]:
    try:
        with open(f"{OUT_DIR}/{fn}") as f:
            results[name] = json.load(f)["test_f1_macro"]
    except FileNotFoundError:
        pass

print("So sánh trên cùng test set (split-by-clip):")
for k, v in sorted(results.items()):
    print(f"  {k:<22}: F1 = {v:.4f}")
print()
if "V1.1 — 34 feat" in results:
    print(f"  V1.2 vs V1.1: {results.get('V1.2 — 56 feat', 0) - results['V1.1 — 34 feat']:+.4f}")
if "V1.2 — 56 feat" in results:
    print(f"  V1.3 vs V1.2: {f1m - results['V1.2 — 56 feat']:+.4f}")

# Đạt ngưỡng đề cương chưa?
print()
if f1m >= 0.80:
    print("🎉 ĐẠT ngưỡng xuất sắc (≥ 0.80)! Tiếp tục V2.0 ST-GCN để có đóng góp học thuật.")
elif f1m >= 0.78:
    print("✓ ĐẠT ngưỡng tối thiểu (≥ 0.78)! Đã đáp ứng đề cương. V2.0 vẫn nên làm để vượt 0.85.")
elif f1m >= 0.72:
    print("~ Cải thiện tốt nhưng chưa đạt 0.78. Có thể train thêm hoặc nhảy V2.0 ST-GCN.")
else:
    print("✗ Cải thiện ít. Vấn đề có thể ở dataset/kiến trúc, chuyển V2.0 ST-GCN.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: training curves
axes[0].plot(hist.history['accuracy'], label='train', alpha=0.8)
axes[0].plot(hist.history['val_accuracy'], label='val', alpha=0.8)
axes[0].set_title('V1.3 — Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist.history['loss'], label='train', alpha=0.8)
axes[1].plot(hist.history['val_loss'], label='val', alpha=0.8)
axes[1].set_title('V1.3 — Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Plot 3: confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Normal','Shoplifting'], yticklabels=['Normal','Shoplifting'])
axes[2].set_title(f'V1.3 — Confusion Matrix\\nF1={f1m:.4f}')
axes[2].set_xlabel('Dự đoán'); axes[2].set_ylabel('Thực tế')

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v1.3_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved {OUT_DIR}/v1.3_summary.png")

## 10. Bước tiếp theo

Gửi cho tôi `log_v1.3_56feat_regularized.json` để cập nhật báo cáo:
- **Bảng 3.2 mở rộng**: thêm dòng V1.3 (kèm Δ với V1.2)
- **Phân tích 3.5.1**: thảo luận tác động của Dropout/BiLSTM/Augment/Focal
- **Curve accuracy/loss** sẽ chèn vào báo cáo (Hình 3.x)

Quyết định bước kế tiếp dựa kết quả V1.3:

| F1 V1.3 | Hành động |
|---|---|
| ≥ 0,80 | Đã vượt ngưỡng tối thiểu. Tiếp V2.0 ST-GCN cho đóng góp học thuật. |
| 0,75 – 0,79 | Gần ngưỡng. Có thể V2.0 ST-GCN hoặc V1.4 (thêm confidence channel). |
| 0,72 – 0,74 | Plateau với Keras 56-feat. Chuyển V2.0 (PyTorch + ST-GCN). |
| < 0,72 | Quá thấp — có thể train hỏng (kiểm tra log). |